# 00 · Config & Setup

**Smart Meters in London — Big Data on Azure Databricks (Medallion Lakehouse)**

Defines every path, schema and helper. Every other notebook starts with
`%run ./00_config_and_setup`, so you configure things **once, here**.

### Data source
The pipeline is driven by the **raw half-hourly readings** (`LCLid, tstp, energy`) you
uploaded as 21 block files — daily aggregates and load profiles are *derived* from them.

### Storage modes (set the `source_mode` widget)
* **blob**   — read directly from your Azure Blob/ADLS container via `abfss://`
  (needs the Unity Catalog External Location from `SETUP_blob_access.md`).
* **volume** — read from a UC Volume you uploaded into (fallback; zero extra setup).

> **Medallion note:** `bronze`/`silver`/`gold` are *table* layers (schemas). The blob
> container / Volume is just where the source **files** live — it feeds bronze, it is
> not a 4th layer.

## 1. Parameters — fill these in the widget boxes at the top

In [ ]:
dbutils.widgets.dropdown("source_mode", "blob", ["blob", "volume"], "0. Source mode")
dbutils.widgets.text("catalog", "workspace",            "1. Unity Catalog")
dbutils.widgets.text("schema",  "smartmeter",           "2. Project schema")
dbutils.widgets.text("storage_account", "smartmeterprince", "3. (blob) Storage account name")
dbutils.widgets.text("container",       "bronze",           "4. (blob) Container name")
dbutils.widgets.text("base_folder",     "",                 "5. Sub-folder inside container (empty if files are at root)")

SOURCE_MODE     = dbutils.widgets.get("source_mode").strip()
CATALOG         = dbutils.widgets.get("catalog").strip()
SCHEMA          = dbutils.widgets.get("schema").strip()
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account").strip()
CONTAINER       = dbutils.widgets.get("container").strip()
BASE_FOLDER     = dbutils.widgets.get("base_folder").strip().strip("/")  # e.g. "bronze" (your landing folder)

BRONZE, SILVER, GOLD = f"{SCHEMA}_bronze", f"{SCHEMA}_silver", f"{SCHEMA}_gold"

## 2. Create catalog, schemas, and (for fallback) a Volume

In [ ]:
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
except Exception as e:
    print(f"(Using existing catalog '{CATALOG}') -> {e}")

spark.sql(f"USE CATALOG {CATALOG}")
for s in [SCHEMA, BRONZE, SILVER, GOLD]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{s}")

## 3. Resolve the source base path from `source_mode`

In [ ]:
if SOURCE_MODE == "blob":
    _root = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
else:  # volume fallback
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")
    _root = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

# append an optional sub-folder; empty when your files sit at the container root
SOURCE_BASE = f"{_root}/{BASE_FOLDER}" if BASE_FOLDER else _root

print(f"source_mode = {SOURCE_MODE}")
print(f"SOURCE_BASE = {SOURCE_BASE}")

## 4. Path map — the single source of truth for "where is each input?"
Adjust the right-hand sides if your blob/Volume filenames or folders differ
(verify with the `dbutils.fs.ls(SOURCE_BASE)` cell below).

In [ ]:
# NOTE: use EXACT filenames for the single-file sources. Do NOT glob the 'household'
# folder — acorn_details.csv lives there too and has a different schema, which would
# corrupt the read. Only 'halfhourly' is globbed (its 21 blocks share one schema).
PATHS = {
    "halfhourly":    f"{SOURCE_BASE}/halfhourly/*.csv",                       # 21 blocks -> glob OK
    "households":    f"{SOURCE_BASE}/households/informations_households.csv",  # specific file (acorn shares folder)
    "holidays":      f"{SOURCE_BASE}/calendar/uk_bank_holidays.csv",
    "weather_daily": f"{SOURCE_BASE}/weather/weather_daily_darksky.csv",
}

## 5. Helpers used across the pipeline

In [ ]:
def table(layer: str, name: str) -> str:
    """Fully-qualified table name, e.g. table('bronze','halfhourly')."""
    schema = {"bronze": BRONZE, "silver": SILVER, "gold": GOLD}[layer]
    return f"{CATALOG}.{schema}.{name}"


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path.rstrip("/*"))
        return True
    except Exception:
        return False


def read_csv(path: str, **opts):
    """Read CSV(s) with header + schema inference. Accepts globs for the block folder."""
    reader = (spark.read.format("csv")
              .option("header", True).option("inferSchema", True).option("mode", "PERMISSIVE"))
    for k, v in opts.items():
        reader = reader.option(k, v)
    return reader.load(path)


print("Config loaded:", {"catalog": CATALOG, "mode": SOURCE_MODE})

## 6. Verify the source is reachable
Run this; you should see your files + the `halfhourly` folder. If it errors in **blob**
mode, finish `SETUP_blob_access.md` (External Location) first.

In [ ]:
display(dbutils.fs.ls(SOURCE_BASE))